# Travel desk V3-R1 — Qwen3-1.7B + LoRA (Google Colab)

Golden V3-R1 is FROZEN_ACTIVE. The recorded execution trained for 1,078 steps and saved and reloaded its adapter on a Tesla T4. Its training source was `66d26070ba6b98ca3356e39914282bfc72326c76`; exact executed cell source and environment are in the private evidence. This maintained notebook includes the observed compatibility repairs and matched model conversion.

Use a fresh T4 runtime and run the preflight before data preparation. Keep Golden R1 for testing only. Check an existing run before training again. New training always writes a distinct directory. Download and verify the saved-run archive before exporting. Weights and corpus-derived texts remain private while licensing is unresolved.


## 1. Settings

In [ ]:
# Set these two values, then run the cell.
REPO_URL = "https://github.com/dhrtt990-blip/saudi-dialect-to-msa.git"      # your team's GitHub repository, e.g. "https://github.com/<team>/<repo>.git"
HF_REPO_ID = ""    # optional for local training; do not publish before licence clearance

BASE_MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
BASE_MODEL = "Qwen/Qwen3-1.7B"   # 1.7B parameters: inside the 4B limit of the project
GGUF_NAME = "travel-v3-r1-qwen3-1.7b-q8_0.gguf"
BASE_GGUF_NAME = "travel-v3-r1-qwen3-base-q8_0.gguf"   # the untuned model, converted like the tuned one: the baseline

USE_QLORA = False   # This recorded workflow uses float16 LoRA, not 4-bit loading.
EPOCHS = 2
LEARNING_RATE = 1e-4
MAX_LENGTH = 768    # tokens per training example (system prompt + message + rewrite)
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LORA_R = 16
LORA_ALPHA = 32
SEED = 13
CORPUS_CSV = "data/raw/multidialectal_travel_corpus.csv"   # downloaded in step 4

# The corpus has almost no numbers, so the template pairs are what teaches the model to keep facts.
# These two values make about one training pair in five carry facts. They are a starting point, not a tuned choice.
CORPUS_LIMIT = 8000   # cleaned corpus pairs to use (0 = all of them, about 24,000)
SYNTHETIC_N = 2000    # template pairs with numbers, dates, times, codes and names

assert REPO_URL, "Fill in REPO_URL above."
REPO_REF = "0614b77d183f59154852fbf128449c1b72cfe6cf"  # use the final freeze commit on main after activation, never a historical commit
GOLDEN_PATH = "data/golden_v3_r1_candidate.jsonl"


## 2. GPU, code and libraries

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!git clone {REPO_URL} repo
%cd repo
!git checkout {REPO_REF}
!pip install -q -r requirements-train.txt

# Repair optional Colab packages before importing datasets or PEFT.
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"],check=True)
subprocess.run([sys.executable,"-m","pip","install","pyarrow==23.0.1"],check=True)


## 3. Required active-release preflight

This guard refuses all training until V3-R1 is formally activated on the checked-out repository. Never edit the release manifest locally to bypass it. The final freeze must be committed and reviewed first.


In [ ]:
import subprocess
import sys
import json
import hashlib
from pathlib import Path

manifest_path = Path("review/v3_r1_revision_manifest.json")
assert manifest_path.is_file(), "V3-R1 is not available on this checkout. Do not use V2.2. Use the approved freeze commit after merge."
m = json.loads(manifest_path.read_text(encoding="utf-8"))
assert m.get("status") == "FROZEN_ACTIVE", "V3-R1 is not formally activated. Stop; obtain approval and commit freeze evidence first."
assert m.get("candidate") == GOLDEN_PATH, "Wrong release path"
assert m.get("freeze_commit"), "Missing recorded freeze commit"
assert hashlib.sha256(Path(GOLDEN_PATH).read_bytes()).hexdigest() == m["sha256"], "Wrong golden bytes"
rows = [json.loads(line) for line in Path(GOLDEN_PATH).read_text(encoding="utf-8").splitlines()]
assert [r["id"] for r in rows] == [f"t{i:03d}" for i in range(1, 101)], "Not the travel golden set"
for path in (GOLDEN_PATH, str(manifest_path)):
    assert not subprocess.check_output(["git", "status", "--porcelain", "--", path], text=True).strip(), f"Uncommitted change: {path}"
    subprocess.run(["git", "ls-files", "--error-unmatch", path], check=True, capture_output=True)
subprocess.run(["git", "merge-base", "--is-ancestor", m["freeze_commit"], "HEAD"], check=True)
assert subprocess.check_output(["git", "show", f"{m['freeze_commit']}:{GOLDEN_PATH}"]) == Path(GOLDEN_PATH).read_bytes(), "Freeze commit does not match data"
subprocess.run([sys.executable, "-m", "src.validate_golden", GOLDEN_PATH], check=True)
print("V3-R1 active:", m["sha256"], "freeze:", m["freeze_commit"])
V3_PREFLIGHT_OK = True


## 4. Build the training data

The corpus is "a Multidialectal Parallel Arabic Corpus - LLM" by Khalid Almeman (Zenodo record 17776617; the licence statements are in `docs/DATASET_CARD.md`). The first cell downloads the CSV (18.4 MB) with `scripts/download_corpus.py`, checks its size and MD5 against the values Zenodo publishes, and prints the columns and three pairs. The file has six columns, and every cell of `Saudi Dialect` starts with the label `- Saudi: `, which is removed on loading. If the download is refused, download the CSV by hand from https://zenodo.org/records/17776617, upload it to `data/raw/` and run the cell again: it then checks the uploaded file. The script was written without network access to Zenodo, so this cell is the first time it runs against the real site.

`src.data_prep` then drops repeated sentences and pairs whose MSA side does not carry the same numbers and Latin words as the dialect side, keeps `CORPUS_LIMIT` corpus pairs, adds `SYNTHETIC_N` seeded template pairs, adds any team-written files named `data/team_*.jsonl`, removes every pair that is close to a golden text or shares a distinctive fact with one, and fails if any overlap remains. Download `data/processed_v3_r1/prep_report.json` with the last cell of this step and commit it: it is the proof for the dataset card.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "scripts/download_corpus.py", "--out", CORPUS_CSV], capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-2000:])
assert result.returncode == 0, "the corpus was not downloaded or is not the expected file; read the message above"

In [ ]:
import glob
import subprocess
import sys

assert globals().get("V3_PREFLIGHT_OK") is True, "Run the V3-R1 preflight successfully first."
cmd = [sys.executable, "-m", "src.data_prep",
       "--golden", GOLDEN_PATH,
       "--corpus-csv", CORPUS_CSV,
       "--corpus-limit", str(CORPUS_LIMIT),
       "--synthetic-n", str(SYNTHETIC_N),
       "--seed", str(SEED),
       "--out", "data/processed_v3_r1"]
team_files = sorted(glob.glob("data/team_*.jsonl"))
if team_files:
    cmd += ["--team-jsonl", *team_files]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:], result.stderr[-2000:])
assert result.returncode == 0, "data preparation failed; read the message above"
report = json.loads(Path("data/processed_v3_r1/prep_report.json").read_text(encoding="utf-8"))
assert report.get("golden_sha256") == m["sha256"], "Preparation used the wrong benchmark"
assert report.get("overlap_remaining_after_filter") == 0, "Overlap remains; stop before training"
V3_PREP_OK = True


In [ ]:
# Keep the detailed report private; public aggregate counts are in review/.
from google.colab import files
files.download("data/processed_v3_r1/prep_report.json")


## 5. Load the data and look at what the model will see

Compare the two printouts. The training example should contain the same turn markers as the inference prompt, including the empty `<think>` block that Qwen3 uses when thinking is off. If the training text has no `<think>` block while the inference prompt has one, tell the team before training: the model would be trained in a slightly different format from the one it is served in.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("json", data_files={"train": "data/processed_v3_r1/train.jsonl",
                                      "validation": "data/processed_v3_r1/val.jsonl"})
ds = ds.select_columns(["messages"])
print(ds)

tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
example = ds["train"][0]["messages"]
print("=== one training example as the model sees it ===")
print(tok.apply_chat_template(example, tokenize=False))
print("=== the prompt used at inference (thinking off) ===")
print(tok.apply_chat_template(example[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False))

sample = ds["train"].select(range(min(500, len(ds["train"]))))["messages"]
lengths = [len(tok(tok.apply_chat_template(m, tokenize=False))["input_ids"]) for m in sample]
too_long = sum(n > MAX_LENGTH for n in lengths) / len(lengths)
print(f"token length: max {max(lengths)}, share above MAX_LENGTH: {too_long:.1%}")

## 6. Train the LoRA adapter

The recorded run uses float16 LoRA with assistant-only loss. The cell saves progress and both epoch checkpoints, then saves the final adapter automatically. A positive step proves training started; `TRAINING_SAVED` and a successful saved-adapter reload establish completion. Do not rerun this cell over a valid run.


In [ ]:
assert globals().get("V3_PREP_OK") is True
assert globals().get("V3_PREFLIGHT_OK") is True
import time, platform, traceback, importlib.metadata
from datetime import datetime, timezone
from peft import LoraConfig
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
RUN_DIR = Path("outputs") / ("travel-v3-r1-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
versions = {p: importlib.metadata.version(p) for p in ["torch","transformers","peft","trl","datasets","accelerate","huggingface_hub"]}
provenance = {"code_commit": subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip(), "golden_sha256":m["sha256"], "prep_report":report, "seed":SEED, "versions":versions, "python":platform.python_version(), "platform":platform.platform(), "gpu":torch.cuda.get_device_name(0), "gpu_bytes":torch.cuda.get_device_properties(0).total_memory, "started_utc":datetime.now(timezone.utc).isoformat(), "status":"initializing"}
save_json("provenance.json", provenance)
(RUN_DIR / "requirements-train.lock.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
class EvidenceCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        save_json("train_log_history.json",state.log_history)
        save_json("progress.json",{"global_step":state.global_step,"max_steps":state.max_steps,"epoch":state.epoch,"updated_utc":datetime.now(timezone.utc).isoformat()})
    def on_step_end(self,args,state,control,**kwargs):
        if state.global_step == 1:
            save_json("first_step.json",{"global_step":state.global_step,"utc":datetime.now(timezone.utc).isoformat()})
peft_config = LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.05,target_modules="all-linear",task_type="CAUSAL_LM")
training_args = SFTConfig(output_dir=str(RUN_DIR),assistant_only_loss=True,max_length=MAX_LENGTH,packing=False,learning_rate=LEARNING_RATE,num_train_epochs=EPOCHS,per_device_train_batch_size=BATCH_SIZE,gradient_accumulation_steps=GRAD_ACCUMULATION,lr_scheduler_type="cosine",logging_steps=20,eval_strategy="epoch",save_strategy="epoch",fp16=True,bf16=False,seed=SEED,report_to="none",model_init_kwargs={"dtype":torch.float16,"revision":BASE_MODEL_REVISION})
save_json("training_config.json",training_args.to_dict())
print("RUN_DIR",RUN_DIR,flush=True)
try:
    trainer=SFTTrainer(model=BASE_MODEL,args=training_args,train_dataset=ds["train"],eval_dataset=ds["validation"],peft_config=peft_config,processing_class=tok,callbacks=[EvidenceCallback()])
    trainer.model.print_trainable_parameters()
    batch=next(iter(trainer.get_train_dataloader()))
    assert (batch["labels"] != -100).any().item(), "No assistant labels"
    save_json("label_mask_check.json",{"supervised_tokens":int((batch["labels"]!=-100).sum()),"ignored_tokens":int((batch["labels"]==-100).sum())})
    started=time.monotonic()
    train_result=trainer.train()
    ADAPTER_DIR=str(RUN_DIR / "final_adapter")
    trainer.save_model(ADAPTER_DIR)
    tok.save_pretrained(ADAPTER_DIR)
    trainer.save_state()
    save_json("train_metrics.json",train_result.metrics)
    save_json("train_log_history.json",trainer.state.log_history)
    provenance.update(status="training_saved",finished_utc=datetime.now(timezone.utc).isoformat(),elapsed_seconds=time.monotonic()-started,global_step=trainer.state.global_step,peak_gpu_bytes=torch.cuda.max_memory_allocated())
    save_json("provenance.json",provenance)
    print("TRAINING_SAVED",ADAPTER_DIR,train_result.metrics,flush=True)
except Exception:
    (RUN_DIR / "error.txt").write_text(traceback.format_exc())
    provenance["status"]="failed"
    save_json("provenance.json",provenance)
    raise

## 7. Back up the completed run

Record the exact cached base revision, download all checkpoints and the final adapter, and verify the printed SHA-256 and ZIP integrity locally before continuing. The recorded execution did this before releasing the training model.


In [ ]:
assert provenance["status"] == "training_saved"
import shutil, re
from src.common import sha256_file
from google.colab import files
from transformers.utils.hub import cached_file
cached_config = Path(cached_file(BASE_MODEL,"config.json",revision=BASE_MODEL_REVISION,local_files_only=True))
base_revision = cached_config.parent.name
assert re.fullmatch(r"[0-9a-f]{40}",base_revision), str(cached_config)
snapshots = [p.name for p in cached_config.parent.parent.iterdir() if p.is_dir()]
assert snapshots == [base_revision], snapshots
provenance["base_model_revision"] = base_revision
provenance["base_revision_resolution"] = {"method":"sole local Hugging Face snapshot used by training","config_path":str(cached_config),"snapshots":snapshots}
save_json("provenance.json",provenance)
BACKUP_ZIP = shutil.make_archive("/content/"+RUN_DIR.name+"-training","zip",root_dir=RUN_DIR.parent,base_dir=RUN_DIR.name)
print(BACKUP_ZIP,Path(BACKUP_ZIP).stat().st_size,sha256_file(BACKUP_ZIP))
files.download(BACKUP_ZIP)


## 8. Reload the saved adapter and export matched HF models

Proceed after verifying the local backup. Capture loaded and installed versions separately, because a package installed after import may differ from the module still in memory. Free the training model, reload the saved adapter in a subprocess, run five validation examples, and save the untuned base before merging the adapter. No Golden example is used here.


In [ ]:
assert provenance["status"] == "training_saved"
import gc, hashlib, importlib.metadata
assert provenance["base_model_revision"]
provenance["loaded_module_versions"] = {p:str(getattr(sys.modules[p],"__version__","unknown")) for p in ["torch","transformers","peft","trl","datasets","pyarrow","accelerate"] if p in sys.modules}
provenance["installed_module_versions_after_repair"] = {p:importlib.metadata.version(p) for p in ["torch","transformers","peft","trl","datasets","pyarrow","accelerate"]}
provenance["actual_trainer_rows"] = {"train":len(trainer.train_dataset),"validation":len(trainer.eval_dataset)}
assert len(trainer.train_dataset)==report["train"] and len(trainer.eval_dataset)==report["val"]
training_source = next(c for c in reversed(In) if c.lstrip().startswith('assert globals().get("V3_PREP_OK")') and 'train_result=trainer.train()' in c)
(RUN_DIR/"training_cell.py").write_text(training_source)
provenance["training_cell_sha256"] = hashlib.sha256(training_source.encode()).hexdigest()
trainer.model.config.to_json_file(RUN_DIR/"base_model_config.json")
save_json("provenance.json",provenance)
del trainer, batch
gc.collect()
torch.cuda.empty_cache()
EXPORT_DIR = Path("outputs")/("export-"+RUN_DIR.name)
with (RUN_DIR/"export.log").open("w") as log:
    result = subprocess.run([sys.executable,"-u","scripts/export_r1.py","--run-dir",str(RUN_DIR),"--export-dir",str(EXPORT_DIR)],stdout=log,stderr=subprocess.STDOUT)
print((RUN_DIR/"export.log").read_text()[-6000:])
assert result.returncode == 0


## 9. Private model artifacts

The original notebook contained Hub upload commands. Publication remains disabled because the corpus licence statements conflict. No token or login is required for this private export workflow.


In [ ]:
print("Private adapter saved. Public weight upload is disabled pending licence clarification.")


## 10. Convert both models to Q8_0 with one pinned converter

The base and merged weights pass through the same llama.cpp commit and quantization. A separate environment uses Transformers 5.23.0 to read the tokenizer files written by the training runtime. The attempted 4.57.6 environment could not read the saved extra-special-token format; no output from that failed conversion is used. Standard venv failed in Colab; virtualenv succeeded. Save both conversion logs and the full manifest.


In [ ]:
import subprocess, sys, hashlib
converter_dir = Path("/content/llama-r1")
converter_commit = "bb692efaa43809dd8584fefddda42a54a9a50846"
if not converter_dir.exists():
    converter_dir.mkdir()
    subprocess.run(["git","init",str(converter_dir)],check=True)
    subprocess.run(["git","-C",str(converter_dir),"remote","add","origin","https://github.com/ggml-org/llama.cpp.git"],check=True)
    subprocess.run(["git","-C",str(converter_dir),"fetch","--depth","1","origin",converter_commit],check=True)
    subprocess.run(["git","-C",str(converter_dir),"checkout","--detach",converter_commit],check=True)
assert subprocess.check_output(["git","-C",str(converter_dir),"rev-parse","HEAD"],text=True).strip()==converter_commit
assert hashlib.sha256((converter_dir/"convert_hf_to_gguf.py").read_bytes()).hexdigest()=="297502d115010e97b996e20dc9cdcfc90ee293df9b616259e5a646b64f410630"
venv_dir=Path("/content/r1-converter-virtualenv")
with (RUN_DIR/"converter_environment.log").open("w") as log:
    subprocess.run([sys.executable,"-m","pip","install","virtualenv"],check=True,stdout=log,stderr=subprocess.STDOUT)
    subprocess.run([sys.executable,"-m","virtualenv","--system-site-packages",str(venv_dir)],check=True,stdout=log,stderr=subprocess.STDOUT)
    converter_python=str(venv_dir/"bin/python")
    subprocess.run([converter_python,"-m","pip","install","transformers==5.23.0","numpy==2.2.6","sentencepiece==0.2.1","protobuf==4.25.8"],check=True,stdout=log,stderr=subprocess.STDOUT)
GGUF_DIR=Path("outputs")/("gguf-"+RUN_DIR.name)
GGUF_DIR.mkdir(exist_ok=True)
(GGUF_DIR/"requirements-converter.lock.txt").write_text(subprocess.check_output([converter_python,"-m","pip","freeze"],text=True))
with (RUN_DIR/"conversion_driver.log").open("w") as log:
    result=subprocess.run([converter_python,"scripts/convert_r1.py","--run-dir",str(RUN_DIR),"--export-dir",str(EXPORT_DIR),"--converter",str(converter_dir),"--out",str(GGUF_DIR)],stdout=log,stderr=subprocess.STDOUT)
print((RUN_DIR/"conversion_driver.log").read_text()[-6000:])
assert result.returncode==0


In [ ]:
# The matched conversion manifest includes file sizes and SHA-256 hashes.
conversion = json.loads((GGUF_DIR/"conversion_manifest.json").read_text())
assert conversion["adapter_sha256"] == sha256_file(Path(ADAPTER_DIR)/"adapter_model.safetensors")
print(json.dumps(conversion,indent=2))


In [ ]:
# Download one GGUF at a time. For large browser transfers, split into 512 MiB
# chunks, hash each chunk, then concatenate and verify the full manifest hash.
from google.colab import files
files.download(str(GGUF_DIR/BASE_GGUF_NAME))
# After the first transfer is verified, download GGUF_DIR/GGUF_NAME as well.


In [ ]:
print("GGUF files are private. No upload or publication is performed.")


## 11. Save exact environment and execution evidence

The run already saves installed package versions and pip freeze, and the export cell records versions actually loaded in the notebook. Preserve both records. The successful historical execution had pyarrow 26.0.1 loaded even though 23.0.1 was installed afterward; it is explicitly recorded, not presented as a clean restart with 23.0.1.


In [ ]:
print("Training evidence:",RUN_DIR)
print("Conversion evidence:",GGUF_DIR)
print(json.dumps(json.loads((RUN_DIR/"provenance.json").read_text()),indent=2))


## 12. Evaluation after freeze and successful training

Use the same frozen V3-R1 file and a new results directory. Do not reuse old predictions, judgments or adapter names. First run the release preflight on the frozen checkout, record the data digest and code commit, and create the distinct Ollama models from the matched GGUF files.

```bash
python -m src.evaluate --golden data/golden_v3_r1_candidate.jsonl --base-model travel-v3-r1-base --tuned-model travel-v3-r1-lora --out results/v3_r1
python -m src.evaluate --golden data/golden_v3_r1_candidate.jsonl --from-predictions results/v3_r1 --judgments results/v3_r1/judgments.csv --out results/v3_r1
python -m src.safety_check --inputs data/safety_travel_v3_r1.jsonl --model travel-v3-r1-lora --out results/v3_r1
```

Do not run until the active release preflight has passed. Copy the judgment template for two independent human raters. Do not mix historical V2.2 results into the V3-R1 directory.
